Importing packages


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns 
print("All libraries imported!")

All libraries imported!


Shape of the imported data

In [3]:
df = pd.read_csv(R'C:\Users\vikas\OneDrive\Desktop\ecommerce-analytics-project\data\raw/data.csv', encoding='latin1')
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

Shape: (541909, 8)

First 5 rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


Missing values and nature of the data 

In [4]:
print("column info:")
df.info()
print("\nMissing values:")
print(df.isnull().sum())

column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


Basic stats 


In [5]:
print("Basic Stats:")
print(df.describe())

Basic Stats:


            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.000000
50%         3.000000       2.080000   15152.000000
75%        10.000000       4.130000   16791.000000
max     80995.000000   38970.000000   18287.000000


Quality check

In [6]:
print("Duplicates Rows:",df.duplicated().sum())
print("Negative Quantities:",(df['Quantity']<0).sum())
print("Zero/negative prices:",(df['UnitPrice']<=0).sum())
print("Unique Customers:", df['CustomerID'].nunique())
print("Unique invoices:",df['InvoiceNo'].nunique())
print("Date Range:",df['InvoiceDate'].min(),"to",df['InvoiceDate'].max())

Duplicates Rows: 5268
Negative Quantities: 10624
Zero/negative prices: 2517
Unique Customers: 4372
Unique invoices: 25900
Date Range: 1/10/2011 10:04 to 9/9/2011 9:52


Columns Overview


In [7]:
print("All Columns:",df.columns.tolist())
print("\nSample CustomerIDs:",df['CustomerID'].dropna().unique()[:10])
print("\nSample StockCodes:",df['StockCode'].unique()[:10])

All Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Sample CustomerIDs: [17850. 13047. 12583. 13748. 15100. 15291. 14688. 17809. 15311. 14527.]

Sample StockCodes: ['85123A' '71053' '84406B' '84029G' '84029E' '22752' '21730' '22633'
 '22632' '84879']


Remove Duplicates

In [8]:
print("Before duplicate:", df.shape)
df_clean=df.drop_duplicates()
print("After duplicate:", df_clean.shape)
print(f"Removed:{df.shape[0]-df_clean.shape[0]} rows")

Before duplicate: (541909, 8)
After duplicate: (536641, 8)
Removed:5268 rows


 Handle Cancellations and handle negative quantities 

In [10]:
print("Before Cancellations:", df_clean.shape)
print("Negative quantities:",(df_clean['Quantity']<0).sum())

df_clean=df_clean[df_clean['Quantity']>0].reset_index(drop=True)
print("After cancellations:",df_clean.shape)

Before Cancellations: (536641, 8)
Negative quantities: 10587
After cancellations: (526054, 8)


Remove Invalid Prices

In [13]:
print("Before Price filter:",df_clean.shape)
print("Invalid prices:", (df_clean['UnitPrice']<=0).sum())
df_clean=df_clean[df_clean['UnitPrice']>0].reset_index(drop=True)
print("After price filter:",df_clean.shape)

Before Price filter: (524878, 8)
Invalid prices: 0
After price filter: (524878, 8)


Date Conversion

In [15]:
df_clean['InvoiceDate']=pd.to_datetime(df_clean['InvoiceDate'],format='%m/%d/%Y %H:%M')
print("Date range:",df_clean['InvoiceDate'].min(), "to", df_clean['InvoiceDate'].max())
print("Date dtype:", df_clean['InvoiceDate'].dtype)

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Date dtype: datetime64[ns]


 CustomerID Strategy

In [17]:
# Handle missing CustomerIDs
print("Missing CustomerID before:",df_clean['CustomerID'].isnull().sum())
print("Drop strategy (focus on known customers for RFM):")

# Drop rows with missing CustomerID
df_clean= df_clean.dropna(subset=['CustomerID']).reset_index(drop=True)
print("After drop CustomerID:",df_clean.shape)
print("Missing CustomerID now:",df_clean['CustomerID'].isnull().sum())

Missing CustomerID before: 0
Drop strategy (focus on known customers for RFM):
After drop CustomerID: (392692, 8)
Missing CustomerID now: 0


Create Total Amount

In [ ]:
# Feature Engineering (Unit I: Data Transformation)
df_clean['TotalAmount']=df_clean['Quantity']*df_clean['UnitPrice']
print("New feature created: TotalAmount")
print("Sample:")
df_clean[['Quantity','UnitPrice', 'TotalAmount']].head()

New feature created: TotalAmount
Sample:


,Quantity,UnitPrice,TotalAmount
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


Date Features

In [19]:
# Date features (Unit I: Data Transformation & Discretization)
df_clean['InvoiceYear']=df_clean['InvoiceDate'].dt.year
df_clean['InvoiceMonth']=df_clean['InvoiceDate'].dt.month
df_clean['InvoiceDay']=df_clean['InvoiceDate'].dt.day
df_clean['InvoiceHour']=df_clean['InvoiceDate'].dt.hour
df_clean['DayOfWeek']=df_clean['InvoiceDate'].dt.dayofweek

print("New date features created:")
print(df_clean[['InvoiceYear','InvoiceMonth','InvoiceDay','InvoiceHour','DayOfWeek']].head())

New date features created:
   InvoiceYear  InvoiceMonth  InvoiceDay  InvoiceHour  DayOfWeek
0         2010            12           1            8          2
1         2010            12           1            8          2
2         2010            12           1            8          2
3         2010            12           1            8          2
4         2010            12           1            8          2


Final Shape Summary

In [25]:
print("Final Clean Dataset:")
print(f"Shape:{df_clean.shape}")
print("\nColumns:", df_clean.columns.tolist())
print("\nmemory usage:",df_clean.memory_usage(deep=True).sum()/1024**2,"MB")
print("\nDate Range:",df_clean['InvoiceDate'].min().date(),"to",df_clean['InvoiceDate'].max().date())
print(f"Total revenue: {df_clean['TotalAmount'].sum():,.2f}")

Final Clean Dataset:
Shape:(392692, 14)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TotalAmount', 'InvoiceYear', 'InvoiceMonth', 'InvoiceDay', 'InvoiceHour', 'DayOfWeek']

memory usage: 114.95059585571289 MB

Date Range: 2010-12-01 to 2011-12-09
Total revenue: 8,887,208.89


Save Clean Data

In [28]:
output_path= 'C:/Users/vikas/OneDrive/Desktop/ecommerce-analytics-project/data/processed/clean_data.csv'
df_clean.to_csv(output_path, index=False)
print(f"Clean data saved: {output_path}")
print(f"File size: {round(df_clean.memory_usage(deep=True).sum() / 1024**2, 2)} MB")

# Verify saved file
print("\nVerification:")
df_saved = pd.read_csv(output_path)
print("Saved shape:", df_saved.shape)
print("Columns match:", len(df_saved.columns) == len(df_clean.columns))

Clean data saved: C:/Users/vikas/OneDrive/Desktop/ecommerce-analytics-project/data/processed/clean_data.csv
File size: 114.95 MB

Verification:
Saved shape: (392692, 14)
Columns match: True


Validation Checks or Final Quality Check

In [ ]:
print("CLEANING SUMMARY:")
print(f"Raw data:        {541909:,} rows")
print(f"Clean data:      {df_clean.shape[0]:,} rows ({100*df_clean.shape[0]/541909:.1f}% kept)")
print(f"Removed total:   {541909 - df_clean.shape[0]:,} rows")
print("\n NO MORE ISSUES:")
print("Duplicates:", df_clean.duplicated().sum())
print("Negative Quantity:", (df_clean['Quantity'] < 0).sum())
print("Invalid prices:", (df_clean['UnitPrice'] <= 0).sum())
print("Missing CustomerID:", df_clean['CustomerID'].isnull().sum())
print("\n UNIT I COMPLETE: Data Preparation & Preprocessing ✓")

CLEANING SUMMARY:
Raw data:        541,909 rows
Clean data:      392,692 rows (72.5% kept)
Removed total:   149,217 rows

 NO MORE ISSUES:
Duplicates: 0
Negative Quantity: 0
Invalid prices: 0
Missing CustomerID: 0

 UNIT I COMPLETE: Data Preparation & Preprocessing ✓
